In [ ]:
%cd /home/raman/school/CMPT_459/project/cmpt-459-project

In [ ]:
from feature_selection.multi_feature_explorer import MultiFeatureExplorer
from utils.enums import Ticker, TimeFrame
from datetime import datetime
import matplotlib.pyplot as plt
from feature_selection.base_models import QuantileBinningModel, DecisionTreeBinningModel
from feature_selection.objective_metric import SortinoRatio, SharpeRatio

In [ ]:


explorer = MultiFeatureExplorer.from_bias_node(
    module_name='cmma',             
    ticker=[
        Ticker.ES,    # E-Mini S&P
        Ticker.NQ,    # E-Mini Nasdaq
        Ticker.YM,    # E-Mini Dow
        Ticker.RTY,   # E-Mini Russell 2000

        # # Energy
        # Ticker.CL,    # Crude Oil (WTI)
        # Ticker.HO,    # ULSD (Heating Oil)

        # # Metals
        # Ticker.GC,    # Gold
        # Ticker.HG,    # Copper
        # Ticker.SI,    # Silver
        # Ticker.PL,    # Platinum

        # # # Currencies (FX)
        # Ticker.EU,    # Euro Currency
        # Ticker.JY,    # Japanese Yen
        # Ticker.BP,    # British Pound
        # Ticker.CD,    # Canadian Dollar
        # Ticker.SF,    # Swiss Franc

        # # Agricultural (Food Grains)
        # Ticker.C,     # Corn
        # Ticker.S,     # Soybean
        # Ticker.W,     # Wheat (SRW)

        # # Meat
        # Ticker.GF,    # Feeder Cattle

        # # Fixed Income
        # Ticker.TY,    # 10Yr T.Note
        # Ticker.FV,    # 5Yr T.Note
        # Ticker.US,    # 30Yr T.Bond
        # Ticker.TU,     # 2Yr T.Note
    ],
    params={
        # 'span_fast': 64,
        # 'span_slow': 256,
        # 'span_ewsd': 32,
        # 'forecast_scalar': 1
        'lookback': 2,
        
    },
    start=datetime(2000, 1, 1),
    end=datetime(2024, 12, 31)
)



In [ ]:

# Call methods directly - they apply to ALL features automatically!
figs = explorer.plot_feature_deciles(n_bins=10, target_col='log_return')
explorer.plot_2bin()



# Show all plots
plt.show()

In [ ]:

# Call methods directly - they apply to ALL features automatically!
figs = explorer.plot_feature_deciles(n_bins=10, target_col='log_return_ewsd')

# Show all plots
plt.show()

In [ ]:

# Example 1: Quantile binning with Sortino
model = QuantileBinningModel(n_bins=3, selection_metric='sortino')
metric = SortinoRatio(annualization_factor=252)

results_df = explorer.walkforward_analysis(
    model=model,
    objective_metric=metric,
    train_start=datetime(1995, 1, 1),
    train_end=datetime(2010, 1, 1),
    test_step=365,
    num_steps=14,
    strategy='long',
    target_col='log_return', 
    save_path='walkforward_results.html'
)

# # Example 2: Decision tree with Sharpe
# model = DecisionTreeBinningModel(n_bins=4, min_samples_leaf_pct=0.05)
# metric = SharpeRatio(annualization_factor=252)

# results_df = explorer.walkforward_analysis(
#     model=model,
#     objective_metric=metric,
#     train_start=datetime(1995, 1, 1),
#     train_end=datetime(2010, 1, 1),
#     test_step=365,
#     num_steps=14,
#     strategy='long'
# )

In [ ]:
results_df = explorer.binning_permutation_test(
    n_bins=3,
    min_samples_per_bin_pct=0.05,
    nreps=1000,
    alpha=0.1,
    n_jobs=-1
)

In [ ]:
explorer.plot_rolling_decile_whiskers(
    window_size=5475,  # 1 year windows
    step_size=365,     # Roll forward quarterly
    n_bins=3
)

In [ ]:
results_quantile = explorer.walkforward_bin_selection(
    train_start=datetime(1995, 1, 1),
    train_end=datetime(2010, 1, 1),  # 10 years training
    test_step=365,                    # 1 year test periods
    num_steps=14,                     # Test 2010-2024
    n_bins=3      # 3 bins
)

In [ ]:
results_tree = explorer.walkforward_bin_selection_tree(
    train_start=datetime(1995, 1, 1),
    train_end=datetime(2010, 1, 1),
    test_step=365,
    num_steps=14,
    n_bins=3,
    min_samples_leaf_pct=0.05
)